# THEBE 10k: SFM + True BDConv + Centerline-Guided Refinement (100 Epochs)

This notebook trains a **KAN-free** limited-data fault segmentation model using the existing fixed cache of 10,000 training patches and 2,000 validation patches.

The proposed model combines:

1. a pretrained SFM ViT encoder and MLA decoder for global coarse guidance;
2. full/half/quarter local refinement with the existing Sobel feature guidance;
3. true modulated deformable convolution at quarter and half resolution;
4. an auxiliary centerline head whose probability is fed back through a learnable residual spatial gate;
5. weighted BCE, Dice, stable Sobel-gradient Dice, and centerline supervision.

The cache is opened read-only and is never recreated. All checkpoints and history are written to a new run directory, so earlier experiments are not overwritten.


## 1. Imports and configuration

Training uses staged fine-tuning. The pretrained encoder is frozen for two warm-up epochs, the final three transformer blocks are unfrozen from epoch 3, and the final six blocks are unfrozen from epoch 31. The run may stop before epoch 100 when validation ODS F1 does not improve for 15 epochs.

In [ ]:
from pathlib import Path
import sys, json, random, math, gc, re, copy, platform, importlib, pathlib, types
from contextlib import contextmanager

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import torchvision
    from torchvision.ops import deform_conv2d
except Exception as error:
    raise RuntimeError(
        "True BDConv requires a torchvision build compatible with the installed "
        "PyTorch/CUDA version. Reinstall matching torch and torchvision builds."
    ) from error

# ============================================================
# WINDOWS PATHS -- normally only these two roots need adjustment
# ============================================================
THEBE_ROOT = Path(r"D:\Downloads\dataset ayang")
SFM_ROOT = Path(r"C:\Users\numpppy\Downloads\TORI_fault_mask_bundle")
EXTRACTED_ROOT = THEBE_ROOT

ROOT = SFM_ROOT
SFM_CHECKPOINT = SFM_ROOT / "SFM-Base.pth"
SFM_MODEL_FILE = SFM_ROOT / "models_Segmentation.py"

# A new run folder prevents older KAN and baseline checkpoints from mixing.
RUN_DIR = THEBE_ROOT / "runs" / "thebe_sfm_10k_bdconv_centerline_gate_100ep"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Read-only cache source. Its historical folder name contains "kan", but the
# arrays are only seismic images and binary masks. No KAN module is loaded.
CACHE_DIR = (
    THEBE_ROOT
    / "runs"
    / "thebe_sfm_10k_kan_centerline_gate"
    / "patch_cache"
)

PATCH_SIZE = 96
PATCH_STRIDE = 48
ROI_SAMPLE_START = 800
ROI_SAMPLE_END = 1300

BATCH_SIZE = 64
INFER_BATCH_SIZE = 64
EPOCHS = 100
WARMUP_EPOCHS = 2
STAGE3_START_EPOCH = 31
UNFREEZE_STAGE2_BLOCKS = 3
UNFREEZE_STAGE3_BLOCKS = 6
EARLY_STOPPING_PATIENCE = 15
LR_PLATEAU_PATIENCE = 5

LR_NEW_WARMUP = 3e-4
LR_NEW_STAGE2 = 1e-4
LR_ENCODER_STAGE2 = 1e-5
LR_NEW_STAGE3 = 5e-5
LR_ENCODER_STAGE3 = 5e-6
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

# Loss: primary BCE+Dice contribution is 0.54 and boundary contribution is
# 0.36 (same 60:40 ratio as BDCNet), plus 0.10 auxiliary centerline loss.
LOSS_W_BCE = 0.27
LOSS_W_DICE = 0.27
LOSS_W_BOUNDARY = 0.36
LOSS_W_CENTERLINE = 0.10

THRESHOLDS = np.round(np.arange(0.10, 0.901, 0.05), 2)
SEED = 1
AUTO_RESUME = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SYSTEM_NAME = platform.system()
NUM_WORKERS = 0 if SYSTEM_NAME == "Windows" else 4
PIN_MEMORY = DEVICE.type == "cuda"
NON_BLOCKING = DEVICE.type == "cuda"
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("Operating system :", SYSTEM_NAME)
print("THEBE root       :", THEBE_ROOT)
print("Cache directory  :", CACHE_DIR)
print("Run directory    :", RUN_DIR)
print("SFM checkpoint   :", SFM_CHECKPOINT)
print("Model file       :", SFM_MODEL_FILE)
print("Device           :", DEVICE)
print("Batch size       :", BATCH_SIZE)
print("Maximum epochs   :", EPOCHS)
print("AMP enabled      :", USE_AMP)
print("Torchvision      :", torchvision.__version__)
print("True BDConv      : enabled at quarter and half fusion")
print("Centerline gate  : enabled")

for required_path in (THEBE_ROOT, CACHE_DIR, SFM_CHECKPOINT, SFM_MODEL_FILE):
    if not required_path.exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU              :", props.name)
    print("VRAM             :", round(props.total_memory / 1024**3, 2), "GB")

## 2. Locate seismic and label files

Each seismic file is a block of approximately 100 crosslines with shape `(crossline, inline, sample)`. The label has the same shape. The available `faultval*.npy` and `faulttest*.npy` files are indexed below; the notebook uses an internal validation block when separate validation seismic files are not present.


In [ ]:
def find_seismic_file(index, split):
    candidates = [
        EXTRACTED_ROOT / f"seis{split}{index}.npy",
        THEBE_ROOT / f"seis{split}{index}.npy",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def label_file(index, split):
    return THEBE_ROOT / f"fault{split}{index}.npy"


def make_pairs(split, ids):
    pairs = {}
    for index in ids:
        seismic_path = find_seismic_file(index, split)
        mask_path = label_file(index, split)
        if seismic_path is None or not mask_path.exists():
            raise FileNotFoundError(
                f"Missing pair: {split}{index}; "
                f"seismic={seismic_path}, mask={mask_path}"
            )
        seismic_shape = np.load(
            seismic_path, mmap_mode="r", allow_pickle=False
        ).shape
        mask_shape = np.load(
            mask_path, mmap_mode="r", allow_pickle=False
        ).shape
        if seismic_shape != mask_shape:
            raise ValueError(
                f"Shape mismatch for {split}{index}: "
                f"{seismic_shape} vs {mask_shape}"
            )
        pairs[index] = (index, seismic_path, mask_path)
    return pairs


TRAIN_IDS = list(range(1, 10))
VAL_IDS = list(range(1, 3))
TEST_IDS = list(range(1, 8))

train_pairs = make_pairs("train", TRAIN_IDS)
val_pairs = make_pairs("val", VAL_IDS)
test_pairs = make_pairs("test", TEST_IDS)

print("Train pairs     :", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Test pairs      :", len(test_pairs))


## 3. Paper-style preprocessing utilities

Local arrays are `(crossline, inline, sample)`. Sections are transposed to `(sample, inline)` before patching. The local files already contain 1803 crosslines, so the paper's four unlabeled original crosslines must not be removed again.


In [ ]:
def minmax_normalize(section):
    x = np.asarray(section, dtype=np.float32)

    low = float(np.nanmin(x))
    high = float(np.nanmax(x))

    if not np.isfinite(low) or not np.isfinite(high):
        return np.zeros_like(x, dtype=np.float32)

    if high <= low:
        return np.zeros_like(x, dtype=np.float32)

    x = np.nan_to_num(
        x,
        nan=low,
        posinf=high,
        neginf=low,
    )

    return ((x - low) / (high - low)).astype(np.float32)


def reflect_pad_to_grid(
    array,
    patch=PATCH_SIZE,
    stride=PATCH_STRIDE,
):
    """
    Reflective padding so the sliding-window grid is integer-sized.
    """

    h, w = array.shape

    if h <= patch:
        target_h = patch
    else:
        target_h = (
            patch
            + int(np.ceil((h - patch) / stride)) * stride
        )

    if w <= patch:
        target_w = patch
    else:
        target_w = (
            patch
            + int(np.ceil((w - patch) / stride)) * stride
        )

    pad_h = target_h - h
    pad_w = target_w - w

    if pad_h > 0 or pad_w > 0:
        array = np.pad(
            array,
            (
                (0, pad_h),
                (0, pad_w),
            ),
            mode="reflect",
        )

    return array


def starts(
    length,
    patch=PATCH_SIZE,
    stride=PATCH_STRIDE,
):
    """
    Sliding-window start positions after reflective padding.
    """

    if length <= patch:
        return [0]

    return list(
        range(
            0,
            length - patch + 1,
            stride,
        )
    )


def paper_section(array, crossline):
    """
    Local format:
        (crossline, inline, sample)

    Paper/image format:
        (sample, inline)
    """

    return np.asarray(array[crossline]).T


def prepare_training_section(
    section,
    is_mask=False,
):
    """
    Paper-style training preprocessing.

    Important:
    ROI 800–1300 is NOT applied here.
    Training uses the full 1537-sample section.
    """

    if is_mask:
        section = np.asarray(
            section,
            dtype=np.float32,
        )
    else:
        section = minmax_normalize(section)

    return reflect_pad_to_grid(section)


def prepare_section(
    section,
    is_mask=False,
):
    """
    Evaluation preprocessing.

    ROI 800–1300 is applied only after the
    full crossline has been reconstructed.
    """

    if is_mask:
        section = np.asarray(
            section,
            dtype=np.float32,
        )
    else:
        section = minmax_normalize(section)

    section = section[
        ROI_SAMPLE_START:ROI_SAMPLE_END,
        :,
    ]

    return reflect_pad_to_grid(section)


print("Preprocessing utilities loaded.")
print("Training ROI crop: disabled")
print("Evaluation ROI:", ROI_SAMPLE_START, ROI_SAMPLE_END)

## 4. Direct read-only cache validation

The fixed 10k/2k arrays are opened with memory mapping. This cell does not sample patches, preprocess volumes, create files, or overwrite the existing cache. It also calculates the capped square-root positive-class weight used by the training loss.

In [ ]:
TRAIN_IMAGE_CACHE = CACHE_DIR / "train_10k_valid_seismic_images.npy"
TRAIN_MASK_CACHE = CACHE_DIR / "train_10k_valid_seismic_masks.npy"
VAL_IMAGE_CACHE = CACHE_DIR / "val_2k_valid_seismic_images.npy"
VAL_MASK_CACHE = CACHE_DIR / "val_2k_valid_seismic_masks.npy"


def validate_saved_cache(image_path, mask_path, expected_count, split_name):
    image_path = Path(image_path)
    mask_path = Path(mask_path)
    if not image_path.exists():
        raise FileNotFoundError(f"{split_name} image cache not found: {image_path}")
    if not mask_path.exists():
        raise FileNotFoundError(f"{split_name} mask cache not found: {mask_path}")

    images = np.load(image_path, mmap_mode="r", allow_pickle=False)
    masks = np.load(mask_path, mmap_mode="r", allow_pickle=False)
    expected_shape = (expected_count, 1, PATCH_SIZE, PATCH_SIZE)

    print(f"\n{split_name.upper()} CACHE")
    print("Image path    :", image_path)
    print("Mask path     :", mask_path)
    print("Image shape   :", images.shape)
    print("Mask shape    :", masks.shape)
    print("Expected shape:", expected_shape)
    print("Image dtype   :", images.dtype)
    print("Mask dtype    :", masks.dtype)

    if images.shape != expected_shape or masks.shape != expected_shape:
        raise RuntimeError(
            f"{split_name} cache mismatch. images={images.shape}, "
            f"masks={masks.shape}, expected={expected_shape}. No file was changed."
        )

    print("[OK] Existing cache reused in read-only mode.")
    del images, masks


validate_saved_cache(TRAIN_IMAGE_CACHE, TRAIN_MASK_CACHE, 10_000, "train")
validate_saved_cache(VAL_IMAGE_CACHE, VAL_MASK_CACHE, 2_000, "validation")

train_mask_memmap = np.load(TRAIN_MASK_CACHE, mmap_mode="r", allow_pickle=False)
positive_pixels = float(train_mask_memmap.sum())
negative_pixels = float(train_mask_memmap.size - positive_pixels)
positive_ratio = positive_pixels / max(float(train_mask_memmap.size), 1.0)
RAW_POS_WEIGHT = negative_pixels / max(positive_pixels, 1.0)
POS_WEIGHT = float(np.clip(np.sqrt(RAW_POS_WEIGHT), 1.0, 6.0))
del train_mask_memmap

print("\nTraining positive pixel ratio :", positive_ratio)
print("Raw inverse-frequency weight  :", RAW_POS_WEIGHT)
print("BCE positive weight used      :", POS_WEIGHT)
print("[CACHE-ONLY MODE] Sampling and cache construction skipped.")

The cache has already fixed the 10k training and 2k validation subsets. This baseline intentionally does not create `train_index`, `val_index`, or `test_index`.


## 5. Cached patch dataset

Training augmentation is deliberately moderate: horizontal flip, seismic polarity reversal, mild intensity scaling, and low-amplitude Gaussian noise. Geometric vertical flipping is not used because it reverses the time/depth direction. Validation is deterministic and receives no augmentation.

In [ ]:
class CachedThebePatchDataset(Dataset):
    def __init__(self, image_path, mask_path, augment=False):
        self.image_path = Path(image_path)
        self.mask_path = Path(mask_path)
        self.augment = augment
        self._images = None
        self._masks = None

    def _open(self):
        if self._images is None:
            self._images = np.load(self.image_path, mmap_mode="r", allow_pickle=False)
            self._masks = np.load(self.mask_path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        self._open()
        return len(self._images)

    def __getitem__(self, index):
        self._open()
        image = np.array(self._images[index], dtype=np.float32, copy=True)
        target = np.array(self._masks[index], dtype=np.float32, copy=True)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=2).copy()
                target = np.flip(target, axis=2).copy()
            if random.random() < 0.25:
                image = 1.0 - image
            if random.random() < 0.30:
                scale = random.uniform(0.90, 1.10)
                image = np.clip((image - 0.5) * scale + 0.5, 0.0, 1.0)
            if random.random() < 0.25:
                noise = np.random.normal(0.0, 0.01, size=image.shape).astype(np.float32)
                image = np.clip(image + noise, 0.0, 1.0)

        return torch.from_numpy(image), torch.from_numpy(target)


train_dataset = CachedThebePatchDataset(
    TRAIN_IMAGE_CACHE, TRAIN_MASK_CACHE, augment=True
)
val_dataset = CachedThebePatchDataset(
    VAL_IMAGE_CACHE, VAL_MASK_CACHE, augment=False
)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_dataset, shuffle=True, drop_last=False, **loader_kwargs
)
val_loader = DataLoader(
    val_dataset, shuffle=False, drop_last=False, **loader_kwargs
)

print("Train patches      :", len(train_dataset))
print("Validation patches :", len(val_dataset))
print("Train batches      :", len(train_loader))
print("Validation batches :", len(val_loader))

## 6. Inspect a cached patch

The visualization selects a fault-rich example directly from the cached masks and disables augmentation for deterministic inspection.


In [ ]:
cached_masks = np.load(
    TRAIN_MASK_CACHE, mmap_mode="r", allow_pickle=False
)
cached_ratios = cached_masks.mean(axis=(1, 2, 3))
rich_indices = np.flatnonzero(cached_ratios >= 0.03)
if len(rich_indices) == 0:
    raise RuntimeError("No fault-rich patch exists in the train cache.")

visual_index = int(rich_indices[np.argmax(cached_ratios[rich_indices])])
visual_ratio = float(cached_ratios[visual_index])
del cached_masks, cached_ratios, rich_indices

visual_dataset = CachedThebePatchDataset(
    TRAIN_IMAGE_CACHE, TRAIN_MASK_CACHE, augment=False
)
image, target = visual_dataset[visual_index]

fault_overlay = np.ma.masked_where(target[0].numpy() < 0.5, target[0].numpy())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image[0], cmap="gray", aspect="auto")
axes[0].set_title("Cached 96x96 seismic patch")
axes[1].imshow(image[0], cmap="gray", aspect="auto")
axes[1].imshow(fault_overlay, cmap="autumn", alpha=0.85, aspect="auto")
axes[1].set_title(f"Fault-rich patch (ratio={visual_ratio:.3f})")
for axis in axes:
    axis.axis("off")
plt.tight_layout()


## 7. SFM with boundary-deformable and centerline-guided refinement

The pretrained SFM encoder and MLA decoder provide global coarse fault guidance. A hierarchical CNN branch extracts full-, half-, and quarter-resolution local features. Existing Sobel image guidance is retained for a controlled comparison with the previous run. After ordinary channel fusion, true modulated deformable convolution refines the quarter and half features. The centerline probability is then fed back through a learnable residual spatial gate before the final segmentation head. There is no KAN and no confidence head.


In [ ]:
import copy, importlib, pathlib, types
from contextlib import contextmanager

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

models_segmentation = importlib.import_module("models_Segmentation")
if not hasattr(models_segmentation, "vit_base_patch16"):
    raise AttributeError(f"vit_base_patch16 tidak ditemukan di {SFM_MODEL_FILE}.")


@contextmanager
def pathlib_checkpoint_compatibility():
    original_posix_path = pathlib.PosixPath
    original_windows_path = pathlib.WindowsPath
    try:
        if sys.platform.startswith("win"):
            pathlib.PosixPath = pathlib.WindowsPath
        else:
            pathlib.WindowsPath = pathlib.PosixPath
        yield
    finally:
        pathlib.PosixPath = original_posix_path
        pathlib.WindowsPath = original_windows_path


def load_checkpoint_file(path: Path):
    with pathlib_checkpoint_compatibility():
        try:
            return torch.load(str(path), map_location="cpu", weights_only=False)
        except TypeError:
            return torch.load(str(path), map_location="cpu")


def extract_state_dict(checkpoint) -> dict[str, torch.Tensor]:
    if not isinstance(checkpoint, dict):
        raise TypeError(f"Checkpoint type harus dict, bukan {type(checkpoint)}")
    for container in ("model", "model_state_dict", "state_dict"):
        if isinstance(checkpoint.get(container), dict):
            checkpoint = checkpoint[container]
            break

    cleaned: dict[str, torch.Tensor] = {}
    for original_key, value in checkpoint.items():
        if not torch.is_tensor(value):
            continue
        key = original_key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "model.", "encoder.", "backbone."):
                if key.startswith(prefix):
                    key = key[len(prefix) :]
                    changed = True
        cleaned[key] = value
    if not cleaned:
        raise RuntimeError("Checkpoint tidak mempunyai tensor state_dict.")
    return cleaned


def interpolate_positional_embedding(
    model: nn.Module,
    state: dict[str, torch.Tensor],
) -> None:
    if "pos_embed" not in state or not hasattr(model, "pos_embed"):
        return
    source = state["pos_embed"]
    destination = model.pos_embed
    if source.shape == destination.shape:
        return

    destination_patches = model.patch_embed.num_patches
    extra_tokens = destination.shape[1] - destination_patches
    source_patches = source.shape[1] - extra_tokens
    old_grid = int(round(math.sqrt(source_patches)))
    new_grid = int(round(math.sqrt(destination_patches)))
    if old_grid**2 != source_patches or new_grid**2 != destination_patches:
        print("Skipping incompatible pos_embed:", tuple(source.shape))
        state.pop("pos_embed", None)
        return

    extra = source[:, :extra_tokens]
    patch = source[:, extra_tokens:].reshape(
        1, old_grid, old_grid, source.shape[-1]
    ).permute(0, 3, 1, 2)
    patch = F.interpolate(
        patch,
        size=(new_grid, new_grid),
        mode="bicubic",
        align_corners=False,
    )
    patch = patch.permute(0, 2, 3, 1).reshape(
        1, new_grid * new_grid, source.shape[-1]
    )
    state["pos_embed"] = torch.cat([extra, patch], dim=1)
    print(f"Interpolated pos_embed: {old_grid}x{old_grid} -> {new_grid}x{new_grid}")


def group_count(channels: int, maximum_groups: int = 8) -> int:
    for groups in range(min(maximum_groups, channels), 0, -1):
        if channels % groups == 0:
            return groups
    return 1


def replace_batchnorm_with_groupnorm(module: nn.Module) -> None:
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            setattr(
                module,
                name,
                nn.GroupNorm(group_count(child.num_features), child.num_features),
            )
        else:
            replace_batchnorm_with_groupnorm(child)


base_constructor = getattr(models_segmentation, "vit_base_patch16")
base_model = base_constructor(
    in_chans=1,
    img_size=PATCH_SIZE,
    num_classes=1,
    drop_path_rate=0.10,
)

checkpoint = load_checkpoint_file(SFM_CHECKPOINT)
pretrained_state = extract_state_dict(checkpoint)
interpolate_positional_embedding(base_model, pretrained_state)

model_state = base_model.state_dict()
compatible_encoder_state: dict[str, torch.Tensor] = {}
shape_skipped: dict[str, tuple[tuple[int, ...], tuple[int, ...]]] = {}

for key, value in pretrained_state.items():
    # The SFM checkpoint is used as encoder pretraining only.
    if key.startswith(("decoder.", "segmentation_head.", "head.")):
        continue
    if key not in model_state:
        continue
    expected = model_state[key]
    if (
        key == "patch_embed.proj.weight"
        and value.ndim == 4
        and expected.ndim == 4
        and value.shape[1] == 3
        and expected.shape[1] == 1
    ):
        value = value.mean(dim=1, keepdim=True)
    if value.shape == expected.shape:
        compatible_encoder_state[key] = value
    else:
        shape_skipped[key] = (tuple(value.shape), tuple(expected.shape))

load_result = base_model.load_state_dict(compatible_encoder_state, strict=False)
replace_batchnorm_with_groupnorm(base_model.decoder)


def mla_head_forward_align_false(
    self,
    mla_p2: torch.Tensor,
    mla_p3: torch.Tensor,
    mla_p4: torch.Tensor,
    mla_p5: torch.Tensor,
) -> torch.Tensor:
    target_size = (4 * mla_p2.shape[-2], 4 * mla_p2.shape[-1])
    heads = [
        F.interpolate(
            head(feature),
            size=target_size,
            mode="bilinear",
            align_corners=False,
        )
        for head, feature in zip(
            (self.head2, self.head3, self.head4, self.head5),
            (mla_p2, mla_p3, mla_p4, mla_p5),
        )
    ]
    return torch.cat(heads, dim=1)


def vit_mla_forward_align_false(
    self,
    x1: torch.Tensor,
    x2: torch.Tensor,
    x3: torch.Tensor,
    x4: torch.Tensor,
    h: int = 14,
    w: int = 14,
) -> torch.Tensor:
    batch_size, number_of_patches, hidden = x1.shape
    if h == w:
        inferred_grid = int(round(math.sqrt(number_of_patches)))
        if inferred_grid * inferred_grid == number_of_patches:
            h = w = inferred_grid

    def tokens_to_map(tokens: torch.Tensor) -> torch.Tensor:
        return tokens.permute(0, 2, 1).contiguous().view(
            batch_size, hidden, h, w
        )

    x = self.mlahead(
        tokens_to_map(x1),
        tokens_to_map(x2),
        tokens_to_map(x3),
        tokens_to_map(x4),
    )
    x = self.cls(x)
    return F.interpolate(
        x,
        size=(h * 16, w * 16),
        mode="bilinear",
        align_corners=False,
    )


# The original file uses align_corners=True in two decoder interpolation stages.
# Patch only this model instance; the user's source file is not modified.
base_model.decoder.mlahead.forward = types.MethodType(
    mla_head_forward_align_false,
    base_model.decoder.mlahead,
)
base_model.decoder.forward = types.MethodType(
    vit_mla_forward_align_false,
    base_model.decoder,
)

# This head is defined in models_Segmentation.py but is not called in forward().
for parameter in base_model.segmentation_head.parameters():
    parameter.requires_grad = False



class ConvGNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels, kernel_size=3,
                stride=stride, padding=1, bias=False,
            ),
            nn.GroupNorm(group_count(out_channels), out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class BoundaryDeformableConv2d(nn.Module):
    """Boundary-guided modulated deformable refinement.

    The paper leaves multi-channel boundary handling and f(x, A) unspecified.
    This implementation uses depthwise Sobel-x responses and concatenation of
    x with |Sobel-x(x)| to predict 18 offsets and 9 modulation values for a
    3x3 deformable kernel. Zero offsets and a high initial mask bias provide a
    stable start close to regular convolution. The block is residual so the
    pretrained/local feature path remains available during early training.
    """

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.channels = channels
        self.kernel_size = 3
        self.offset_channels = 2 * self.kernel_size * self.kernel_size
        self.mask_channels = self.kernel_size * self.kernel_size

        sobel_x = torch.tensor(
            [[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]],
            dtype=torch.float32,
        ).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", sobel_x, persistent=False)

        self.offset_mask = nn.Conv2d(
            2 * channels,
            self.offset_channels + self.mask_channels,
            kernel_size=3,
            padding=1,
        )
        nn.init.zeros_(self.offset_mask.weight)
        nn.init.zeros_(self.offset_mask.bias)
        with torch.no_grad():
            # sigmoid(2) ~= 0.88, closer to an initially open modulation mask.
            self.offset_mask.bias[self.offset_channels:].fill_(2.0)

        self.weight = nn.Parameter(
            torch.empty(channels, channels, self.kernel_size, self.kernel_size)
        )
        self.bias = nn.Parameter(torch.zeros(channels))
        nn.init.kaiming_normal_(self.weight, mode="fan_out", nonlinearity="relu")

        self.norm = nn.GroupNorm(group_count(channels), channels)
        self.activation = nn.SiLU(inplace=True)
        self.residual_scale_logit = nn.Parameter(torch.tensor(-2.0))

    def boundary_attention(self, x: torch.Tensor) -> torch.Tensor:
        kernel = self.sobel_x.to(device=x.device, dtype=x.dtype).expand(
            self.channels, 1, 3, 3
        )
        boundary = F.conv2d(x, kernel, padding=1, groups=self.channels).abs()
        scale = boundary.amax(dim=(2, 3), keepdim=True).clamp_min(1e-6)
        return boundary / scale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        boundary = self.boundary_attention(x)
        offset_mask = self.offset_mask(torch.cat([x, boundary], dim=1))
        offset = offset_mask[:, :self.offset_channels]
        mask = torch.sigmoid(offset_mask[:, self.offset_channels:])

        deformed = deform_conv2d(
            input=x,
            offset=offset,
            weight=self.weight,
            bias=self.bias,
            stride=(1, 1),
            padding=(1, 1),
            dilation=(1, 1),
            mask=mask,
        )
        deformed = self.activation(self.norm(deformed))
        residual_scale = torch.sigmoid(self.residual_scale_logit)
        return x + residual_scale * deformed


class SobelBoundaryGuidance(nn.Module):
    """Fixed Sobel gradients with learnable multi-scale feature gates."""

    def __init__(self, full_channels=32, half_channels=48, quarter_channels=64):
        super().__init__()
        sobel_x = torch.tensor(
            [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]
        ).view(1, 1, 3, 3)
        sobel_y = torch.tensor(
            [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]
        ).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", sobel_x, persistent=False)
        self.register_buffer("sobel_y", sobel_y, persistent=False)
        self.full_gate = nn.Conv2d(1, full_channels, kernel_size=1)
        self.half_gate = nn.Conv2d(1, half_channels, kernel_size=1)
        self.quarter_gate = nn.Conv2d(1, quarter_channels, kernel_size=1)

    def magnitude(self, image: torch.Tensor) -> torch.Tensor:
        gx = F.conv2d(image, self.sobel_x, padding=1)
        gy = F.conv2d(image, self.sobel_y, padding=1)
        magnitude = torch.sqrt(gx.square() + gy.square() + 1e-6)
        maximum = magnitude.amax(dim=(2, 3), keepdim=True).clamp_min(1e-6)
        return magnitude / maximum

    @staticmethod
    def apply_gate(feature: torch.Tensor, edge: torch.Tensor, projection: nn.Module):
        edge = F.interpolate(
            edge, size=feature.shape[-2:], mode="bilinear", align_corners=False
        )
        gate = torch.sigmoid(projection(edge))
        return feature * (1.0 + 0.5 * gate)

    def forward(self, image, full_feature, half_feature, quarter_feature):
        edge = self.magnitude(image)
        return (
            self.apply_gate(full_feature, edge, self.full_gate),
            self.apply_gate(half_feature, edge, self.half_gate),
            self.apply_gate(quarter_feature, edge, self.quarter_gate),
        )


class BoundaryAwareHybridRefinement(nn.Module):
    """SFM-guided refinement with BDConv and centerline spatial feedback."""

    def __init__(self, full_channels=32, half_channels=48, quarter_channels=64):
        super().__init__()
        self.full_stem = nn.Sequential(
            ConvGNAct(1, full_channels),
            ConvGNAct(full_channels, full_channels),
        )
        self.half_encoder = nn.Sequential(
            ConvGNAct(full_channels, half_channels, stride=2),
            ConvGNAct(half_channels, half_channels),
        )
        self.quarter_encoder = nn.Sequential(
            ConvGNAct(half_channels, quarter_channels, stride=2),
            ConvGNAct(quarter_channels, quarter_channels),
        )
        self.boundary_guidance = SobelBoundaryGuidance(
            full_channels, half_channels, quarter_channels
        )
        self.coarse_guidance = nn.Sequential(
            ConvGNAct(1, quarter_channels),
            ConvGNAct(quarter_channels, quarter_channels),
        )
        self.quarter_fusion = nn.Sequential(
            ConvGNAct(2 * quarter_channels, quarter_channels),
            ConvGNAct(quarter_channels, quarter_channels),
        )
        self.quarter_bdconv = BoundaryDeformableConv2d(quarter_channels)
        self.half_fusion = nn.Sequential(
            ConvGNAct(quarter_channels + half_channels, half_channels),
            ConvGNAct(half_channels, half_channels),
        )
        self.half_bdconv = BoundaryDeformableConv2d(half_channels)
        self.full_fusion = nn.Sequential(
            ConvGNAct(half_channels + full_channels, full_channels),
            ConvGNAct(full_channels, full_channels),
        )
        self.segmentation_head = nn.Sequential(
            ConvGNAct(full_channels, full_channels),
            nn.Conv2d(full_channels, 1, kernel_size=1),
        )
        center_channels = max(16, full_channels // 2)
        self.centerline_head = nn.Sequential(
            ConvGNAct(full_channels, center_channels),
            nn.Conv2d(center_channels, 1, kernel_size=1),
        )
        self.centerline_gate = nn.Sequential(
            ConvGNAct(full_channels + 1, full_channels),
            nn.Conv2d(full_channels, 1, kernel_size=1),
        )
        # A small initial residual scale avoids suppressing pretrained features.
        self.centerline_gate_scale_logit = nn.Parameter(torch.tensor(-2.0))

    def forward(self, image: torch.Tensor, coarse_logits: torch.Tensor):
        full_feature = self.full_stem(image)
        half_feature = self.half_encoder(full_feature)
        quarter_feature = self.quarter_encoder(half_feature)

        full_feature, half_feature, quarter_feature = self.boundary_guidance(
            image, full_feature, half_feature, quarter_feature
        )

        coarse_quarter = F.interpolate(
            coarse_logits,
            size=quarter_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        coarse_feature = self.coarse_guidance(coarse_quarter)
        quarter_fused = self.quarter_fusion(
            torch.cat([quarter_feature, coarse_feature], dim=1)
        )
        quarter_fused = self.quarter_bdconv(quarter_fused)

        quarter_to_half = F.interpolate(
            quarter_fused,
            size=half_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        half_fused = self.half_fusion(
            torch.cat([half_feature, quarter_to_half], dim=1)
        )
        half_fused = self.half_bdconv(half_fused)

        half_to_full = F.interpolate(
            half_fused,
            size=full_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        full_fused = self.full_fusion(
            torch.cat([full_feature, half_to_full], dim=1)
        )

        centerline_logits = self.centerline_head(full_fused)
        centerline_probability = torch.sigmoid(centerline_logits)
        spatial_gate = torch.sigmoid(
            self.centerline_gate(
                torch.cat([full_fused, centerline_probability], dim=1)
            )
        )
        centered_gate = 2.0 * spatial_gate - 1.0
        gate_scale = torch.sigmoid(self.centerline_gate_scale_logit)
        guided_feature = full_fused * (1.0 + gate_scale * centered_gate)
        segmentation_logits = self.segmentation_head(guided_feature)
        return segmentation_logits, centerline_logits


class SFMHybridHighResolutionDecoder(nn.Module):
    def __init__(self, base: nn.Module) -> None:
        super().__init__()
        self.base = base
        self.refinement = BoundaryAwareHybridRefinement()

    def forward(self, image: torch.Tensor, return_aux: bool = False):
        coarse_logits = self.base(image)
        if isinstance(coarse_logits, (tuple, list)):
            coarse_logits = coarse_logits[-1]
        if coarse_logits.shape[-2:] != image.shape[-2:]:
            coarse_logits = F.interpolate(
                coarse_logits,
                size=image.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        segmentation_logits, centerline_logits = self.refinement(image, coarse_logits)
        if return_aux:
            return segmentation_logits, centerline_logits
        return segmentation_logits


if not compatible_encoder_state:
    raise RuntimeError(
        "Tidak ada tensor pretrained encoder yang kompatibel; training dihentikan."
    )
if not any(key.startswith("blocks.") for key in compatible_encoder_state):
    raise RuntimeError(
        "Transformer block pretrained tidak ter-load; periksa format SFM-Base.pth."
    )

MODEL = SFMHybridHighResolutionDecoder(base_model).to(DEVICE)

try:
    with torch.inference_mode():
        smoke_input = torch.randn(1, 1, PATCH_SIZE, PATCH_SIZE, device=DEVICE)
        smoke_segmentation, smoke_centerline = MODEL(smoke_input, return_aux=True)
except RuntimeError as error:
    raise RuntimeError(
        "BDConv smoke test failed. Verify that torchvision matches the installed "
        "PyTorch and CUDA builds."
    ) from error

for output_name, output_tensor in (
    ("segmentation", smoke_segmentation),
    ("centerline", smoke_centerline),
):
    if output_tensor.shape != smoke_input.shape:
        raise RuntimeError(
            f"Smoke test {output_name} gagal: input={tuple(smoke_input.shape)}, "
            f"output={tuple(output_tensor.shape)}"
        )
    if not torch.isfinite(output_tensor).all():
        raise FloatingPointError(f"Smoke test {output_name} non-finite.")

loaded_encoder_parameters = sum(
    model_state[key].numel() for key in compatible_encoder_state
)
refinement_parameters = sum(p.numel() for p in MODEL.refinement.parameters())
has_kan = any("kan" in type(module).__name__.lower() for module in MODEL.modules())

print("Pretrained encoder load : SAFE name + shape matching")
print("Loaded encoder tensors  :", len(compatible_encoder_state))
print("Loaded encoder params   :", f"{loaded_encoder_parameters:,}")
print("Missing base-model keys :", len(load_result.missing_keys))
print("Shape-skipped keys      :", len(shape_skipped))
print("Refinement params       :", f"{refinement_parameters:,}")
print("Smoke input             :", tuple(smoke_input.shape))
print("Smoke segmentation      :", tuple(smoke_segmentation.shape))
print("Smoke centerline        :", tuple(smoke_centerline.shape))
print("Quarter BDConv          :", type(MODEL.refinement.quarter_bdconv).__name__)
print("Half BDConv             :", type(MODEL.refinement.half_bdconv).__name__)
print("Centerline spatial gate :", type(MODEL.refinement.centerline_gate).__name__)
print("All model parameters    :", f"{sum(p.numel() for p in MODEL.parameters()):,}")
print("KAN modules detected    :", has_kan)
if has_kan:
    raise RuntimeError("KAN module detected although this experiment must be KAN-free.")

del smoke_input, smoke_segmentation, smoke_centerline
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_id = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(gpu_id)

    total_vram = props.total_memory / 1024**3
    allocated = torch.cuda.memory_allocated(gpu_id) / 1024**3
    reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3

    print("GPU:", props.name)
    print(f"Total VRAM     : {total_vram:.2f} GB")
    print(f"Allocated VRAM : {allocated:.2f} GB")
    print(f"Reserved VRAM  : {reserved:.2f} GB")
    print(f"Free estimate  : {total_vram - reserved:.2f} GB")
else:
    print("CUDA tidak tersedia.")

## 8. Stable mixed boundary-aware objective and metrics

Weighted BCE and Dice supervise fault regions, while a stable gradient-Dice objective aligns Sobel boundaries. The squared-gradient denominator keeps the overlap well behaved for unnormalized Sobel magnitudes. The thin THEBE fault trace is also used to supervise the auxiliary centerline head. This objective is a documented adaptation of the BDCNet idea rather than a claim of exact reproduction of its ambiguous Equation 9.


In [ ]:
POS_WEIGHT_TENSOR = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)


def weighted_bce_loss(logits, target):
    return F.binary_cross_entropy_with_logits(
        logits, target.float(), pos_weight=POS_WEIGHT_TENSOR
    )


def soft_dice_loss(logits, target, eps=1e-6):
    probability = torch.sigmoid(logits)
    target = target.float()
    dims = (1, 2, 3)
    intersection = (probability * target).sum(dim=dims)
    denominator = probability.sum(dim=dims) + target.sum(dim=dims)
    return (1.0 - (2.0 * intersection + eps) / (denominator + eps)).mean()


SOBEL_X = torch.tensor(
    [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
    dtype=torch.float32,
    device=DEVICE,
).view(1, 1, 3, 3)
SOBEL_Y = torch.tensor(
    [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],
    dtype=torch.float32,
    device=DEVICE,
).view(1, 1, 3, 3)


def sobel_magnitude(tensor):
    gx = F.conv2d(tensor, SOBEL_X.to(dtype=tensor.dtype), padding=1)
    gy = F.conv2d(tensor, SOBEL_Y.to(dtype=tensor.dtype), padding=1)
    return torch.sqrt(gx.square() + gy.square() + 1e-6)


def boundary_overlap_loss(logits, target, eps=1e-6):
    """Stable soft Dice overlap between prediction and target gradients."""
    pred_gradient = sobel_magnitude(torch.sigmoid(logits))
    true_gradient = sobel_magnitude(target.float())
    dims = (1, 2, 3)
    intersection = (pred_gradient * true_gradient).sum(dim=dims)
    denominator = (
        pred_gradient.square().sum(dim=dims)
        + true_gradient.square().sum(dim=dims)
    )
    similarity = (2.0 * intersection + eps) / (denominator + eps)
    return (1.0 - similarity).mean()


def total_loss(seg_logits, centerline_logits, target):
    bce = weighted_bce_loss(seg_logits, target)
    dice = soft_dice_loss(seg_logits, target)
    boundary = boundary_overlap_loss(seg_logits, target)
    centerline = F.binary_cross_entropy_with_logits(
        centerline_logits, target.float(), pos_weight=POS_WEIGHT_TENSOR
    )
    total = (
        LOSS_W_BCE * bce
        + LOSS_W_DICE * dice
        + LOSS_W_BOUNDARY * boundary
        + LOSS_W_CENTERLINE * centerline
    )
    parts = {
        "bce": float(bce.detach()),
        "dice": float(dice.detach()),
        "boundary": float(boundary.detach()),
        "centerline": float(centerline.detach()),
    }
    return total, parts


def confusion_from_probability(probability, target, threshold=0.5):
    pred = probability >= threshold
    true = target >= 0.5
    return {
        "tp": int((pred & true).sum().item()),
        "tn": int((~pred & ~true).sum().item()),
        "fp": int((pred & ~true).sum().item()),
        "fn": int((~pred & true).sum().item()),
    }


def metrics_from_counts(counts):
    tp, tn, fp, fn = counts["tp"], counts["tn"], counts["fp"], counts["fn"]
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
    }

## 9. Staged fine-tuning for up to 100 epochs

The freezing strategy is intentionally identical to the previous run:

- **Epochs 1-2:** freeze the pretrained SFM encoder; train the MLA decoder, BDConv refinement, centerline gate, and output heads.
- **Epochs 3-30:** unfreeze the final three transformer blocks.
- **Epochs 31-100:** unfreeze the final six transformer blocks at a smaller learning rate.

Every epoch writes only to the new run directory. The latest checkpoint supports interruption-safe resume, and the best checkpoint is selected by validation ODS F1 over thresholds 0.10-0.90.


In [ ]:
def split_parameter_groups(model):
    encoder_parameters = []
    new_parameters = []
    for name, parameter in model.named_parameters():
        if name.startswith("base.") and not name.startswith("base.decoder."):
            encoder_parameters.append(parameter)
        else:
            new_parameters.append(parameter)
    return encoder_parameters, new_parameters


ENCODER_PARAMETERS, NEW_PARAMETERS = split_parameter_groups(MODEL)

optimizer = torch.optim.AdamW(
    [
        {"params": ENCODER_PARAMETERS, "lr": LR_ENCODER_STAGE2, "name": "encoder"},
        {"params": NEW_PARAMETERS, "lr": LR_NEW_WARMUP, "name": "new"},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=LR_PLATEAU_PATIENCE,
    min_lr=[1e-7, 1e-6],
)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def desired_stage(epoch):
    if epoch <= WARMUP_EPOCHS:
        return 1
    if epoch < STAGE3_START_EPOCH:
        return 2
    return 3


def configure_stage(stage, reset_learning_rates=True):
    # Freeze every pretrained base component first. The MLA decoder is a new
    # task-specific module and therefore remains trainable in every stage.
    for name, parameter in MODEL.base.named_parameters():
        parameter.requires_grad = name.startswith("decoder.")

    blocks = list(MODEL.base.blocks)
    if stage >= 2:
        count = UNFREEZE_STAGE2_BLOCKS if stage == 2 else UNFREEZE_STAGE3_BLOCKS
        for block in blocks[-count:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        if hasattr(MODEL.base, "norm"):
            for parameter in MODEL.base.norm.parameters():
                parameter.requires_grad = True

    for parameter in MODEL.refinement.parameters():
        parameter.requires_grad = True

    if reset_learning_rates:
        if stage == 1:
            optimizer.param_groups[0]["lr"] = 0.0
            optimizer.param_groups[1]["lr"] = LR_NEW_WARMUP
        elif stage == 2:
            optimizer.param_groups[0]["lr"] = LR_ENCODER_STAGE2
            optimizer.param_groups[1]["lr"] = LR_NEW_STAGE2
        else:
            optimizer.param_groups[0]["lr"] = LR_ENCODER_STAGE3
            optimizer.param_groups[1]["lr"] = LR_NEW_STAGE3

    trainable = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
    total = sum(p.numel() for p in MODEL.parameters())
    print(
        f"[STAGE {stage}] trainable={trainable:,}/{total:,} "
        f"({100.0 * trainable / total:.2f}%) | "
        f"lr_encoder={optimizer.param_groups[0]['lr']:.2e} | "
        f"lr_new={optimizer.param_groups[1]['lr']:.2e}"
    )


BEST_CHECKPOINT_PATH = RUN_DIR / "best_model_bdconv_centerline_gate_10k.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last_model_bdconv_centerline_gate_10k.pt"
HISTORY_PATH = RUN_DIR / "training_history_bdconv_centerline_gate_10k.csv"
CHECKPOINT_PATH = BEST_CHECKPOINT_PATH  # used by later evaluation cells

best_score = -float("inf")
best_epoch = 0
stale_epochs = 0
history = []
start_epoch = 1
active_stage = None

if AUTO_RESUME and LAST_CHECKPOINT_PATH.exists():
    try:
        resume_checkpoint = torch.load(
            str(LAST_CHECKPOINT_PATH), map_location=DEVICE, weights_only=False
        )
    except TypeError:
        resume_checkpoint = torch.load(str(LAST_CHECKPOINT_PATH), map_location=DEVICE)

    MODEL.load_state_dict(resume_checkpoint["model_state_dict"])
    optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    if "scheduler_state_dict" in resume_checkpoint:
        scheduler.load_state_dict(resume_checkpoint["scheduler_state_dict"])
    if "scaler_state_dict" in resume_checkpoint:
        scaler.load_state_dict(resume_checkpoint["scaler_state_dict"])
    best_score = float(resume_checkpoint.get("best_score", -float("inf")))
    best_epoch = int(resume_checkpoint.get("best_epoch", 0))
    stale_epochs = int(resume_checkpoint.get("stale_epochs", 0))
    history = list(resume_checkpoint.get("history", []))
    start_epoch = int(resume_checkpoint["epoch"]) + 1
    active_stage = int(resume_checkpoint.get("active_stage", desired_stage(start_epoch)))
    configure_stage(active_stage, reset_learning_rates=False)
    print(f"[RESUME] Continuing from epoch {start_epoch}")


for epoch in range(start_epoch, EPOCHS + 1):
    stage = desired_stage(epoch)
    if stage != active_stage:
        active_stage = stage
        configure_stage(stage, reset_learning_rates=True)

    MODEL.train()
    train_sum = 0.0
    component_sums = {"bce": 0.0, "dice": 0.0, "boundary": 0.0, "centerline": 0.0}

    for images, targets in tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} train",
        mininterval=5,
        leave=False,
    ):
        images = images.to(DEVICE, non_blocking=NON_BLOCKING)
        targets = targets.to(DEVICE, non_blocking=NON_BLOCKING)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            segmentation, centerline = MODEL(images, return_aux=True)
            loss, loss_parts = total_loss(segmentation, centerline, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in MODEL.parameters() if p.requires_grad], GRAD_CLIP
        )
        scaler.step(optimizer)
        scaler.update()

        train_sum += float(loss.detach())
        for key in component_sums:
            component_sums[key] += loss_parts[key]

    MODEL.eval()
    val_sum = 0.0
    threshold_counts = {
        float(t): {"tp": 0, "tn": 0, "fp": 0, "fn": 0} for t in THRESHOLDS
    }

    with torch.inference_mode():
        for images, targets in tqdm(
            val_loader,
            desc=f"Epoch {epoch}/{EPOCHS} validation",
            mininterval=5,
            leave=False,
        ):
            images = images.to(DEVICE, non_blocking=NON_BLOCKING)
            targets = targets.to(DEVICE, non_blocking=NON_BLOCKING)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                segmentation, centerline = MODEL(images, return_aux=True)
                val_loss_batch, _ = total_loss(segmentation, centerline, targets)
            val_sum += float(val_loss_batch.detach())
            probability = torch.sigmoid(segmentation.float())
            for threshold in THRESHOLDS:
                key = float(threshold)
                batch_counts = confusion_from_probability(probability, targets, key)
                for count_name in threshold_counts[key]:
                    threshold_counts[key][count_name] += batch_counts[count_name]

    train_loss = train_sum / max(len(train_loader), 1)
    val_loss = val_sum / max(len(val_loader), 1)
    metrics_by_threshold = {
        threshold: metrics_from_counts(counts)
        for threshold, counts in threshold_counts.items()
    }
    ods_threshold, ods_metrics = max(
        metrics_by_threshold.items(), key=lambda item: item[1]["f1"]
    )
    fixed_metrics = metrics_by_threshold[min(metrics_by_threshold, key=lambda t: abs(t - 0.5))]
    scheduler.step(ods_metrics["f1"])

    row = {
        "epoch": epoch,
        "stage": active_stage,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_ods_threshold": ods_threshold,
        "val_ods_precision": ods_metrics["precision"],
        "val_ods_recall": ods_metrics["recall"],
        "val_ods_f1": ods_metrics["f1"],
        "val_ods_iou": ods_metrics["iou"],
        "val_f1_at_0.5": fixed_metrics["f1"],
        "val_iou_at_0.5": fixed_metrics["iou"],
        "lr_encoder": optimizer.param_groups[0]["lr"],
        "lr_new": optimizer.param_groups[1]["lr"],
    }
    history.append(row)

    improved = ods_metrics["f1"] > best_score + 1e-6
    next_best_score = ods_metrics["f1"] if improved else best_score
    next_best_epoch = epoch if improved else best_epoch
    next_stale = 0 if improved else stale_epochs + 1

    payload = {
        "epoch": epoch,
        "model_state_dict": MODEL.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "metrics": row,
        "best_score": next_best_score,
        "best_epoch": next_best_epoch,
        "stale_epochs": next_stale,
        "active_stage": active_stage,
        "history": history,
        "configuration": {
            "architecture": "sfm_true_bdconv_centerline_guided_refinement_no_kan",
            "true_bdconv_resolutions": ["quarter", "half"],
            "centerline_spatial_gate": True,
            "boundary_loss": "stable_gradient_dice",
            "train_cache_count": len(train_dataset),
            "validation_cache_count": len(val_dataset),
            "batch_size": BATCH_SIZE,
            "maximum_epochs": EPOCHS,
            "selection_metric": "validation_ods_f1",
            "pos_weight": POS_WEIGHT,
            "loss_weights": {
                "bce": LOSS_W_BCE,
                "dice": LOSS_W_DICE,
                "boundary": LOSS_W_BOUNDARY,
                "centerline": LOSS_W_CENTERLINE,
            },
        },
    }
    torch.save(payload, LAST_CHECKPOINT_PATH)

    try:
        import pandas as pd
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    except Exception as error:
        print("[WARNING] Could not write history CSV:", error)

    print(
        f"Epoch {epoch:03d} | stage={active_stage} | train={train_loss:.4f} | "
        f"val={val_loss:.4f} | ODS_t={ods_threshold:.2f} | "
        f"P={ods_metrics['precision']:.4f} | R={ods_metrics['recall']:.4f} | "
        f"F1={ods_metrics['f1']:.4f} | IoU={ods_metrics['iou']:.4f} | "
        f"lr_enc={row['lr_encoder']:.2e} | lr_new={row['lr_new']:.2e}"
    )

    if improved:
        best_score = ods_metrics["f1"]
        best_epoch = epoch
        stale_epochs = 0
        torch.save(payload, BEST_CHECKPOINT_PATH)
        print("Best checkpoint saved:", BEST_CHECKPOINT_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping at epoch {epoch}. "
                f"Best epoch={best_epoch}, best validation ODS F1={best_score:.4f}"
            )
            break

print("Training complete.")
print("Best validation ODS F1:", best_score)
print("Best epoch            :", best_epoch)
print("Best checkpoint       :", BEST_CHECKPOINT_PATH)
print("Latest checkpoint     :", LAST_CHECKPOINT_PATH)

## 10. Plot training history


In [ ]:
import pandas as pd

history_df = pd.DataFrame(history)
display(history_df)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["val_ods_f1"], label="ODS F1")
axes[1].plot(history_df["epoch"], history_df["val_ods_iou"], label="ODS IoU")
axes[1].set_title("Validation metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["val_ods_threshold"])
axes[2].set_title("Validation ODS threshold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0.0, 1.0)

plt.tight_layout()

## 11. Load the best model before stitched evaluation

The checkpoint chosen by validation ODS F1 is loaded before reconstructing full sections. Subsequent evaluation uses the paper-aligned ROI (samples 800-1300), overlapping 96×96 windows, smooth weighted stitching, and validation threshold selection.

## 11. Sliding-window reconstruction and paper-style metrics

Inference runs over every 96x96 window in the complete sample 800–1300 ROI, including windows that were filtered from training. Overlapping predictions are averaged to reconstruct each crossline. Validation reconstruction determines the global ODS threshold; test results are then reported with AP/PR-AUC, ODS, OIS, precision, recall, F1, and IoU.


In [ ]:
try:
    best_checkpoint = torch.load(
        str(CHECKPOINT_PATH), map_location=DEVICE, weights_only=False
    )
except TypeError:
    best_checkpoint = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)

MODEL.load_state_dict(best_checkpoint["model_state_dict"])
MODEL.to(DEVICE)
MODEL.eval()
print("Loaded best checkpoint:", CHECKPOINT_PATH)
print("Saved epoch           :", best_checkpoint.get("epoch"))
print("Saved metrics         :", best_checkpoint.get("metrics"))

from sklearn.metrics import average_precision_score

INFER_BATCH_SIZE = BATCH_SIZE
EVAL_THRESHOLDS = np.linspace(0.05, 0.95, 19)


def smooth_patch_weight(size=PATCH_SIZE):
    """Non-zero raised-cosine weights reduce seams between overlapping tiles."""
    one_dimensional = np.hanning(size + 2)[1:-1].astype(np.float32)
    one_dimensional = np.clip(one_dimensional, 0.05, None)
    weight = np.outer(one_dimensional, one_dimensional)
    weight /= weight.max()
    return weight.astype(np.float32)


SMOOTH_WEIGHT = smooth_patch_weight()


def stitched_crossline(model, seismic_path, mask_path, crossline):
    seismic = np.load(seismic_path, mmap_mode="r", allow_pickle=False)
    mask = np.load(mask_path, mmap_mode="r", allow_pickle=False)
    image_section = prepare_section(paper_section(seismic, crossline), is_mask=False)
    target_section = prepare_section(paper_section(mask, crossline), is_mask=True)
    height, width = image_section.shape
    patch_specs = [(top, left) for top in starts(height) for left in starts(width)]
    prediction_sum = np.zeros((height, width), dtype=np.float32)
    prediction_weight = np.zeros((height, width), dtype=np.float32)

    model.eval()
    with torch.inference_mode():
        for begin in range(0, len(patch_specs), INFER_BATCH_SIZE):
            batch_specs = patch_specs[begin:begin + INFER_BATCH_SIZE]
            batch = np.stack([
                image_section[top:top + PATCH_SIZE, left:left + PATCH_SIZE]
                for top, left in batch_specs
            ]).astype(np.float32)[:, None]

            tensor = torch.from_numpy(batch).to(DEVICE, non_blocking=NON_BLOCKING)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(tensor)
            probabilities = torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()

            for probability, (top, left) in zip(probabilities, batch_specs):
                prediction_sum[
                    top:top + PATCH_SIZE, left:left + PATCH_SIZE
                ] += probability * SMOOTH_WEIGHT
                prediction_weight[
                    top:top + PATCH_SIZE, left:left + PATCH_SIZE
                ] += SMOOTH_WEIGHT

    prediction = prediction_sum / np.maximum(prediction_weight, 1e-6)
    roi_height = ROI_SAMPLE_END - ROI_SAMPLE_START
    prediction = prediction[:roi_height, :width]
    target = target_section[:roi_height, :width]
    return prediction, target.astype(np.uint8)


def iter_stitched_split(model, pairs, crossline_step=1):
    for volume_id, (_, seismic_path, mask_path) in pairs.items():
        mask_shape = np.load(mask_path, mmap_mode="r", allow_pickle=False).shape
        for crossline in range(0, mask_shape[0], crossline_step):
            yield (
                volume_id,
                crossline,
                stitched_crossline(model, seismic_path, mask_path, crossline),
            )


def counts_for_arrays(probability, target, threshold):
    pred = probability >= float(threshold)
    true = target >= 0.5
    return {
        "tp": int(np.logical_and(pred, true).sum()),
        "tn": int(np.logical_and(~pred, ~true).sum()),
        "fp": int(np.logical_and(pred, ~true).sum()),
        "fn": int(np.logical_and(~pred, true).sum()),
    }


def add_counts(destination, source):
    for key in destination:
        destination[key] += source[key]


def evaluate_stitched_split(
    model,
    pairs,
    thresholds=EVAL_THRESHOLDS,
    crossline_step=1,
):
    global_counts = {
        float(t): {"tp": 0, "tn": 0, "fp": 0, "fn": 0}
        for t in thresholds
    }
    per_line_best_f1 = []
    per_line_ap = []

    iterator = iter_stitched_split(model, pairs, crossline_step=crossline_step)
    for _, _, (probability, target) in tqdm(iterator, desc="Stitched crosslines"):
        flat_target = target.reshape(-1)
        flat_probability = probability.reshape(-1)
        if np.unique(flat_target).size > 1:
            per_line_ap.append(
                float(average_precision_score(flat_target, flat_probability))
            )

        best_line_f1 = 0.0
        for threshold in thresholds:
            counts = counts_for_arrays(probability, target, threshold)
            add_counts(global_counts[float(threshold)], counts)
            best_line_f1 = max(
                best_line_f1,
                metrics_from_counts(counts)["f1"],
            )
        per_line_best_f1.append(best_line_f1)

    rows = [
        {
            "threshold": float(threshold),
            **metrics_from_counts(global_counts[float(threshold)]),
        }
        for threshold in thresholds
    ]
    table = pd.DataFrame(rows)
    ods_row = table.loc[table["f1"].idxmax()]
    return {
        "threshold_table": table,
        "ods_threshold": float(ods_row["threshold"]),
        "ods_metrics": ods_row.to_dict(),
        "ois_f1": float(np.mean(per_line_best_f1)),
        "pr_auc": float(np.mean(per_line_ap)) if per_line_ap else float("nan"),
    }


print("Starting stitched validation evaluation...")
val_eval = evaluate_stitched_split(MODEL, val_pairs, crossline_step=1)
display(val_eval["threshold_table"])
print("Validation PR-AUC:", val_eval["pr_auc"])
print("Validation ODS threshold:", val_eval["ods_threshold"])
print("Validation ODS metrics:", val_eval["ods_metrics"])
print("Validation OIS F1:", val_eval["ois_f1"])
BEST_THRESHOLD = val_eval["ods_threshold"]

In [ ]:
TEST_EVAL_CROSSLINE_STEP = 5  # matches the speed-oriented sampling in An et al.

print("Starting stitched test evaluation...")
test_eval = evaluate_stitched_split(
    MODEL,
    test_pairs,
    crossline_step=TEST_EVAL_CROSSLINE_STEP,
)
test_ods = test_eval["ods_metrics"]
test_at_val_threshold = {"threshold": BEST_THRESHOLD}
test_counts = {"tp": 0, "tn": 0, "fp": 0, "fn": 0}

iterator = iter_stitched_split(
    MODEL,
    test_pairs,
    crossline_step=TEST_EVAL_CROSSLINE_STEP,
)
for _, _, (probability, target) in tqdm(
    iterator,
    desc="Test metrics at validation threshold",
):
    add_counts(
        test_counts,
        counts_for_arrays(probability, target, BEST_THRESHOLD),
    )

test_at_val_threshold.update(metrics_from_counts(test_counts))

print("Test PR-AUC:", test_eval["pr_auc"])
print("Test ODS threshold (diagnostic only):", test_eval["ods_threshold"])
print("Test ODS metrics (diagnostic only):", test_ods)
print("Test OIS F1 (diagnostic only):", test_eval["ois_f1"])
print("PRIMARY TEST RESULT -- validation threshold:", test_at_val_threshold)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

# ============================================================
# 1. Load best checkpoint
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

MODEL.load_state_dict(checkpoint["model_state_dict"])
MODEL.to(DEVICE)
MODEL.eval()

print("Loaded checkpoint:", CHECKPOINT_PATH)

# ============================================================
# 2. Visualization settings
# ============================================================

VIS_SPLIT = "test"          # "train", "val", atau "test"
VIS_VOLUME_ID = 1
VIS_CROSSLINE = 50          # index dimulai dari 0

if VIS_SPLIT == "train":
    pairs_for_visualization = train_pairs
elif VIS_SPLIT == "val":
    pairs_for_visualization = val_pairs
elif VIS_SPLIT == "test":
    pairs_for_visualization = test_pairs
else:
    raise ValueError(
        "VIS_SPLIT harus 'train', 'val', atau 'test'"
    )

_, seismic_path, mask_path = pairs_for_visualization[
    VIS_VOLUME_ID
]

# ============================================================
# 3. Load one crossline
# ============================================================

seismic_volume = np.load(
    seismic_path,
    mmap_mode="r",
    allow_pickle=False,
)

fault_volume = np.load(
    mask_path,
    mmap_mode="r",
    allow_pickle=False,
)

# Original local format:
# (crossline, inline, sample)
#
# Paper visualization format:
# (sample, inline)

raw_section = paper_section(
    seismic_volume,
    VIS_CROSSLINE,
)

fault_section = paper_section(
    fault_volume,
    VIS_CROSSLINE,
)

# ============================================================
# 4. Prepare visualization ROI
# ============================================================

# Normalize seismic without reflective padding.
# Padding is only needed for patch extraction/inference.
normalized_full = minmax_normalize(
    raw_section
)

normalized_roi = normalized_full[
    ROI_SAMPLE_START:ROI_SAMPLE_END,
    :,
]

target_roi = np.asarray(
    fault_section[
        ROI_SAMPLE_START:ROI_SAMPLE_END,
        :
    ],
    dtype=np.float32,
)

# ============================================================
# 5. Sliding-window inference and stitching
# ============================================================

prediction_probability, stitched_target = stitched_crossline(
    MODEL,
    seismic_path,
    mask_path,
    VIS_CROSSLINE,
)

prediction_probability = np.asarray(
    prediction_probability,
    dtype=np.float32,
)

stitched_target = np.asarray(
    stitched_target,
    dtype=np.float32,
)

# ============================================================
# Shape alignment: remove right-side padding
# ============================================================

original_height, original_width = normalized_roi.shape

# Prediction and stitched target may contain padding
# introduced for sliding-window inference.
prediction_probability = prediction_probability[
    :original_height,
    :original_width,
]

stitched_target = stitched_target[
    :original_height,
    :original_width,
]

# Crop visualization seismic to exactly the same size
normalized_roi = normalized_roi[
    :original_height,
    :original_width,
]

target_roi = target_roi[
    :original_height,
    :original_width,
]

height, width = prediction_probability.shape

print("Normalized seismic shape:", normalized_roi.shape)
print("Ground-truth shape      :", stitched_target.shape)
print("Prediction shape        :", prediction_probability.shape)

assert normalized_roi.shape == prediction_probability.shape
assert stitched_target.shape == prediction_probability.shape

# ============================================================
# 7. Binary prediction
# ============================================================

threshold = float(BEST_THRESHOLD)

prediction_binary = (
    prediction_probability >= threshold
).astype(np.float32)

# Geological coordinate system:
# horizontal axis = inline
# vertical axis   = sample/time
extent = [
    0,
    width,
    ROI_SAMPLE_END,
    ROI_SAMPLE_START,
]

# ============================================================
# 8. Visualization
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(20, 10),
    constrained_layout=True,
)

# ------------------------------------------------------------
# Panel 1: normalized seismic
# ------------------------------------------------------------

axes[0, 0].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[0, 0].set_title(
    f"Normalized seismic — {VIS_SPLIT}, "
    f"volume {VIS_VOLUME_ID}, "
    f"crossline {VIS_CROSSLINE}"
)
axes[0, 0].set_xlabel("Inline")
axes[0, 0].set_ylabel("Sample / Time")

# ------------------------------------------------------------
# Panel 2: ground-truth fault
# ------------------------------------------------------------

axes[0, 1].imshow(
    stitched_target,
    cmap="gray",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[0, 1].set_title("Ground-truth fault")
axes[0, 1].set_xlabel("Inline")
axes[0, 1].set_ylabel("Sample / Time")

# ------------------------------------------------------------
# Panel 3: predicted probability
# ------------------------------------------------------------

probability_image = axes[0, 2].imshow(
    prediction_probability,
    cmap="magma",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[0, 2].set_title(
    f"Stitched predicted probability\n"
    f"threshold = {threshold:.2f}"
)
axes[0, 2].set_xlabel("Inline")
axes[0, 2].set_ylabel("Sample / Time")

fig.colorbar(
    probability_image,
    ax=axes[0, 2],
    fraction=0.046,
    pad=0.04,
    label="Fault probability",
)

# ------------------------------------------------------------
# Panel 4: binary prediction mask
# ------------------------------------------------------------

axes[1, 0].imshow(
    prediction_binary,
    cmap="gray",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[1, 0].set_title(
    f"Prediction mask\nthreshold = {threshold:.2f}"
)
axes[1, 0].set_xlabel("Inline")
axes[1, 0].set_ylabel("Sample / Time")

# ------------------------------------------------------------
# Panel 5: ground-truth overlay
# ------------------------------------------------------------

axes[1, 1].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[1, 1].contour(
    stitched_target,
    levels=[0.5],
    colors="lime",
    linewidths=0.8,
    origin="upper",
    extent=extent,
)

axes[1, 1].set_title("Ground-truth overlay")
axes[1, 1].set_xlabel("Inline")
axes[1, 1].set_ylabel("Sample / Time")

# ------------------------------------------------------------
# Panel 6: prediction overlay
# ------------------------------------------------------------

axes[1, 2].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="upper",
    extent=extent,
)

axes[1, 2].contour(
    prediction_binary,
    levels=[0.5],
    colors="red",
    linewidths=0.8,
    origin="upper",
    extent=extent,
)

axes[1, 2].set_title(
    f"Prediction overlay\nthreshold = {threshold:.2f}"
)
axes[1, 2].set_xlabel("Inline")
axes[1, 2].set_ylabel("Sample / Time")

# Keep sample/time direction consistent
for ax in axes.ravel():
    ax.set_ylim(
        ROI_SAMPLE_END,
        ROI_SAMPLE_START,
    )

plt.show()

# ============================================================
# 9. Numerical summary
# ============================================================

ground_truth_pixels = int(
    np.sum(stitched_target >= 0.5)
)

predicted_fault_pixels = int(
    np.sum(prediction_binary >= 0.5)
)

print("Visualization summary")
print("---------------------")
print("Split:", VIS_SPLIT)
print("Volume ID:", VIS_VOLUME_ID)
print("Crossline:", VIS_CROSSLINE)
print("ROI shape:", normalized_roi.shape)
print(
    "Sample range:",
    ROI_SAMPLE_START,
    "to",
    ROI_SAMPLE_END,
)
print(
    "Ground-truth fault pixels:",
    ground_truth_pixels,
)
print(
    "Predicted fault pixels:",
    predicted_fault_pixels,
)
print(
    "Maximum probability:",
    float(prediction_probability.max()),
)
print("Threshold:", threshold)

In [ ]:
# ============================================================
# Correctly aligned crossline visualization
# ============================================================

threshold = float(BEST_THRESHOLD)

prediction_binary = (
    prediction_probability >= threshold
)

height, width = prediction_probability.shape

# Use one consistent coordinate system
extent = [
    0,
    width,
    ROI_SAMPLE_START,
    ROI_SAMPLE_END,
]


def format_axis(ax):
    ax.set_xlabel("Inline")
    ax.set_ylabel("Sample / Time")

    # Conventional seismic display:
    # smaller sample/time value at the top
    ax.set_ylim(
        ROI_SAMPLE_END,
        ROI_SAMPLE_START,
    )


fig, axes = plt.subplots(
    2,
    3,
    figsize=(20, 10),
    constrained_layout=True,
)


# 1. Normalized seismic
axes[0, 0].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[0, 0].set_title(
    f"Normalized seismic — {VIS_SPLIT}, "
    f"volume {VIS_VOLUME_ID}, "
    f"crossline {VIS_CROSSLINE}"
)

format_axis(axes[0, 0])


# 2. Ground truth
axes[0, 1].imshow(
    stitched_target,
    cmap="gray",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[0, 1].set_title("Ground-truth fault")
format_axis(axes[0, 1])


# 3. Predicted probability
probability_image = axes[0, 2].imshow(
    prediction_probability,
    cmap="magma",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[0, 2].set_title(
    f"Stitched predicted probability "
    f"(threshold={threshold:.2f})"
)

format_axis(axes[0, 2])

fig.colorbar(
    probability_image,
    ax=axes[0, 2],
    fraction=0.046,
    pad=0.04,
)


# 4. Prediction mask
axes[1, 0].imshow(
    prediction_binary,
    cmap="gray",
    vmin=0,
    vmax=1,
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[1, 0].set_title("Prediction mask")
format_axis(axes[1, 0])


# 5. Ground-truth overlay
axes[1, 1].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[1, 1].contour(
    stitched_target,
    levels=[0.5],
    colors="lime",
    linewidths=0.8,
    extent=extent,
)

axes[1, 1].set_title("Ground-truth overlay")
format_axis(axes[1, 1])


# 6. Prediction overlay
axes[1, 2].imshow(
    normalized_roi,
    cmap="gray",
    aspect="auto",
    origin="lower",
    extent=extent,
)

axes[1, 2].contour(
    prediction_binary.astype(np.float32),
    levels=[0.5],
    colors="red",
    linewidths=0.8,
    extent=extent,
)

axes[1, 2].set_title("Prediction overlay")
format_axis(axes[1, 2])


plt.show()

## Method interpretation

- **An et al. (2021):** THEBE orientation, 96x96 patches, stride 48, ROI 800-1300, overlapping reconstruction, and ODS/OIS/AP-style evaluation.
- **BDCNet-inspired component:** boundary-aware modulated deformable sampling is adapted at quarter and half resolution. Because the paper omits several implementation details, this notebook explicitly documents its depthwise Sobel-x, concatenation, mask initialization, and residual design choices.
- **Centerline contribution:** the auxiliary centerline probability is fed back as a learnable residual spatial gate before segmentation.
- **Limited-data setting:** the pretrained SFM and proposed refinement are trained using only the fixed 10k labeled cache.


## Reproducibility notes

- The architecture contains no KAN and no confidence head.
- The historical cache-folder name contains `kan`, but its `.npy` files contain only normalized seismic patches and masks.
- Cache arrays are opened read-only; patches are never resampled or rewritten.
- The freezing stages are identical to the previous 100-epoch notebook.
- The new run directory prevents every earlier checkpoint and history file from being overwritten.
- `last_model_bdconv_centerline_gate_10k.pt` resumes interrupted training.
- `best_model_bdconv_centerline_gate_10k.pt` is selected by validation ODS F1 and used for stitched test evaluation.
